In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ.setdefault('AWS_DEFAULT_REGION', 'us-west-2')
_os.environ.setdefault('AWS_REGION', 'us-west-2')
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            # Honor sessions that were handed explicit credentials
            # (e.g. notebooks that assume_role into other accounts):
            # clobbering them would silently run every cross-account
            # client as the ambient identity. Bare/SDK-created sessions
            # (no explicit creds) still get the refreshable shared-file
            # creds so long-running waits survive credential rotation.
            _ex = getattr(self, '_credentials', None)
            if _ex is not None:
                return _ex
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


# Recipe Override with SFTTrainer

This notebook demonstrates the **recipe override** feature for fine-tuning with `SFTTrainer`.

The recipe system supports a **3-level precedence** for configuration:

1. **Overrides dict** (highest priority) — passed directly to `SFTTrainer`
2. **Recipe YAML file** — a reusable configuration file
3. **SDK defaults** (lowest priority) — built-in defaults

This means you can define a base recipe in a YAML file and selectively override specific values at runtime.

> **Note:** Actual execution requires valid AWS credentials and an appropriate IAM role. This notebook shows the pattern and API usage.

In [2]:
# Install sagemaker-train from the local staging directory
import sys
import os
sys.path.insert(0, '../../sagemaker-train/src')
sys.path.insert(0, '../../sagemaker-core/src')

# Point botocore at the bundled service model (includes CustomizationTechnique/Peft fields)
os.environ['AWS_DATA_PATH'] = os.path.abspath('../../sagemaker-core/sample')

# Required dependencies
!pip install -q omegaconf pyyaml

## Step 1: Create a Recipe YAML File

A recipe file defines training hyperparameters and configuration in a reusable YAML format.
Let's write one to disk.

In [3]:
import yaml
import os

# Define the recipe as a Python dict
recipe_config = {
    "training": {
        "learning_rate": 1e-5,
        "num_epochs": 3,
        "batch_size": 8,
        "sequence_length": 2048
    }
}

# Write the recipe to a local YAML file
recipe_path = "my_sft_recipe.yaml"
with open(recipe_path, "w") as f:
    yaml.dump(recipe_config, f, default_flow_style=False)

print(f"Recipe written to: {os.path.abspath(recipe_path)}")
print("\nContents:")
with open(recipe_path) as f:
    print(f.read())

Recipe written to: /opt/ml/processing/input/my_sft_recipe.yaml

Contents:
training:
  batch_size: 8
  learning_rate: 1.0e-05
  num_epochs: 3
  sequence_length: 2048



## Step 2: Use the Recipe with SFTTrainer and Apply Overrides

We pass both a `recipe` file and an `overrides` dict to `SFTTrainer`.
The overrides will take precedence over recipe file values.

In [4]:
from sagemaker.core.resources import ModelPackageGroup
from sagemaker.train import SFTTrainer
from sagemaker.train.common import TrainingType
from sagemaker.train.configs import StoppingCondition

# Create SFTTrainer with recipe file + overrides
sft_trainer = SFTTrainer(accept_eula=True, 
    model="nova-textgeneration-lite-v2",
    training_type=TrainingType.LORA,
    training_dataset="s3://notebook-test-engine-ds-674622101542-usw2/datasets/sample-converse-messages.jsonl",
    model_package_group=ModelPackageGroup.create(model_package_group_name="my-fine-tuned-models"),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600),
    # Recipe file provides the base configuration
    recipe="my_sft_recipe.yaml",
    # Overrides selectively replace values from the recipe
    overrides={
        "training_config": {
            "learning_rate": 5e-6,   # Override: lower learning rate
            "max_steps": 20          # Override: more steps
        }
    }
)

print("SFTTrainer configured with recipe + overrides.")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


[07/20/26 18:10:38] INFO     Creating model_package_group resource.                              ]8;id=12556567;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12556568;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#23250\23250]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=12556575;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=12556576;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#361\361]8;;\

                    INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=12556582;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=12556583;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#375\375]8;;\

[07/20/26 18:10:39] INFO     SageMaker session not provided. Using default Session.                  ]8;id=12556590;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=12556591;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

SFTTrainer configured with recipe + overrides.


## Step 3: Inspect the Resolved Configuration

Use `get_resolved_recipe()` to see the final merged configuration.
This shows exactly what will be used during training, after all precedence rules are applied.

In [5]:
# Inspect the final merged configuration
resolved = sft_trainer.get_resolved_recipe()

print("Resolved recipe (after merging):")
print("=" * 40)
for section, params in resolved.items():
    print(f"\n[{section}]")
    if isinstance(params, dict):
        for key, value in params.items():
            print(f"  {key}: {value}")
    else:
        print(f"  {params}")

[07/20/26 18:10:40] INFO     SageMaker session not provided. Using default Session.                  ]8;id=12556596;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=12556597;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

                    WARNING  Override key 'training' from user recipe (my_sft_recipe.yaml)   ]8;id=12556604;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/recipe_resolver.py\recipe_resolver.py]8;;\:]8;id=12556605;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/recipe_resolver.py#549\549]8;;\
                             does not exist in the recipe and will be dropped.                                     

Resolved recipe (after merging):

[run]
  name: my-lora-sft-run-64qsr
  model_type: amazon.nova-2-lite-v1:0:256k
  model_name_or_path: nova-lite-2/prod
  data_s3_path: s3://notebook-test-engine-ds-674622101542-usw2/datasets/sample-converse-messages.jsonl
  validation_data_s3_path: 
  validation_data_s3_path_list: {}
  replicas: 4
  output_s3_path: None
  mlflow_tracking_uri: 
  mlflow_experiment_name: my-lora-sft-experiment
  mlflow_run_name: my-lora-sft-run

[training_config]
  max_steps: 20
  save_steps: 10
  save_top_k: 5
  max_length: 32768
  global_batch_size: 32
  reasoning_enabled: True
  val_check_interval: 2500
  limit_val_batches: 2
  lr_scheduler: {'warmup_steps': 15, 'min_lr': 1e-06}
  optim_config: {'lr': 5e-06, 'weight_decay': 0.0, 'adam_beta1': 0.9, 'adam_beta2': 0.95}
  peft: {'peft_scheme': 'lora', 'lora_tuning': {'alpha': 64, 'lora_plus_lr_ratio': 64.0}}
  model_importance_score: {'fine_tuned_model': 1.0}


## Step 4: Verify Precedence

Let's explicitly verify the 3-level precedence:

| Parameter | Recipe File | Override | Resolved (Used) | Source |
|-----------|------------|----------|-----------------|--------|
| `learning_rate` | 1e-5 | 5e-6 | **5e-6** | Override wins |
| `max_steps` | 10 (default) | 20 | **20** | Override wins |
| `batch_size` | 8 | (not set) | **8** | Recipe file |
| `sequence_length` | 2048 | (not set) | **2048** | Recipe file |

In [6]:
# Programmatic verification of precedence
resolved = sft_trainer.get_resolved_recipe()

training_config = resolved.get("training_config", resolved.get("training", {}))

print("Resolved training_config:")
for key, value in training_config.items():
    print(f"  {key}: {value}")

# Overrides win over recipe file values
assert training_config["optim_config"]["lr"] == 5e-6, "Override should win"
assert training_config["max_steps"] == 20, "Override should win"

print("\nAll precedence checks passed!")

[07/20/26 18:10:41] INFO     SageMaker session not provided. Using default Session.                  ]8;id=12556610;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=12556611;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

Resolved training_config:
  max_steps: 20
  save_steps: 10
  save_top_k: 5
  max_length: 32768
  global_batch_size: 32
  reasoning_enabled: True
  val_check_interval: 2500
  limit_val_batches: 2
  lr_scheduler: {'warmup_steps': 15, 'min_lr': 1e-06}
  optim_config: {'lr': 5e-06, 'weight_decay': 0.0, 'adam_beta1': 0.9, 'adam_beta2': 0.95}
  peft: {'peft_scheme': 'lora', 'lora_tuning': {'alpha': 64, 'lora_plus_lr_ratio': 64.0}}
  model_importance_score: {'fine_tuned_model': 1.0}

All precedence checks passed!


## Step 4b: Override Non-Spec Keys (Power User)

The full Hub recipe template contains many more parameters than what the UI exposes.
Power users can override **any** key in the full recipe — not just the UI-surfaced subset.

Examples of non-spec keys in the real recipe:
- `max_length` — maximum sequence length for training
- `save_top_k` — number of checkpoint saves to keep
- `optim_config.adam_beta1` — optimizer momentum parameter
- `lr_scheduler.warmup_steps` — learning rate warmup steps

Use `sft_trainer.hyperparameters._full_recipe_template` to inspect all available keys.


In [7]:
from sagemaker.core.resources import ModelPackageGroup
# Override non-spec keys that exist in the full recipe template
sft_trainer_power = SFTTrainer(accept_eula=True, 
    model="nova-textgeneration-lite-v2",
    training_type=TrainingType.LORA,
    training_dataset="s3://notebook-test-engine-ds-674622101542-usw2/datasets/sample-converse-messages.jsonl",
    model_package_group=ModelPackageGroup.create(model_package_group_name="my-fine-tuned-models"),
    # Override keys that are in the full recipe but NOT in the UI spec
    overrides={
        "training_config": {
            "max_length": 16384,             # Not in UI spec, but in full recipe
            "save_top_k": 3,                 # Not in UI spec, but in full recipe
            "optim_config": {
                "adam_beta1": 0.85            # Nested non-spec key
            },
            "learning_rate": 5e-6,            # In UI spec (also works)
        }
    }
)

# Inspect resolved recipe — non-spec keys are included
resolved_power = sft_trainer_power.get_resolved_recipe()

print("Resolved recipe (power user overrides):")
print("=" * 50)
training_config = resolved_power.get("training_config", {})
for key, value in sorted(training_config.items()):
    if isinstance(value, dict):
        print(f"  {key}:")
        for k2, v2 in value.items():
            print(f"    {k2}: {v2}")
    else:
        print(f"  {key}: {value}")

# Verify non-spec keys are present and overridden
assert training_config["max_length"] == 16384, "Non-spec key override should work"
assert training_config["save_top_k"] == 3, "Non-spec key override should work"
assert training_config["optim_config"]["adam_beta1"] == 0.85, "Nested non-spec override should work"
assert training_config["optim_config"]["adam_beta2"] == 0.95, "Unchanged default preserved"
print("All power user overrides applied successfully!")


                    INFO     Creating model_package_group resource.                              ]8;id=12556616;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12556617;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#23250\23250]8;;\

                    INFO     SageMaker session not provided. Using default Session.                  ]8;id=12556622;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=12556623;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

[07/20/26 18:10:42] INFO     SageMaker session not provided. Using default Session.                  ]8;id=12556628;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=12556629;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

Resolved recipe (power user overrides):
  global_batch_size: 32
  limit_val_batches: 2
  lr_scheduler:
    warmup_steps: 15
    min_lr: 1e-06
  max_length: 16384
  max_steps: 10
  model_importance_score:
    fine_tuned_model: 1.0
  optim_config:
    lr: 5e-06
    weight_decay: 0.0
    adam_beta1: 0.85
    adam_beta2: 0.95
  peft:
    peft_scheme: lora
    lora_tuning: {'alpha': 64, 'lora_plus_lr_ratio': 64.0}
  reasoning_enabled: True
  save_steps: 10
  save_top_k: 3
  val_check_interval: 2500
All power user overrides applied successfully!


## Step 5: The "No Recipe" Path (Hyperparameters Property)

If you prefer not to use recipe files at all, you can still set hyperparameters directly
using the `hyperparameters` property. This path continues to work as before.

In [8]:
from sagemaker.core.resources import ModelPackageGroup
# Alternative: No recipe file, just direct hyperparameters
sft_trainer_simple = SFTTrainer(accept_eula=True, 
    model="nova-textgeneration-lite-v2",
    training_type=TrainingType.LORA,
    training_dataset="s3://notebook-test-engine-ds-674622101542-usw2/datasets/sample-converse-messages.jsonl",
    model_package_group=ModelPackageGroup.create(model_package_group_name="my-fine-tuned-models"),
)

# Set hyperparameters directly (existing approach, still fully supported)
sft_trainer_simple.hyperparameters.learning_rate = 2e-5

print("SFTTrainer configured without recipe (hyperparameters property).")
print(f"  learning_rate: {sft_trainer_simple.hyperparameters.learning_rate}")

                    INFO     Creating model_package_group resource.                              ]8;id=12556634;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12556635;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#23250\23250]8;;\

[07/20/26 18:10:43] INFO     SageMaker session not provided. Using default Session.                  ]8;id=12556640;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=12556641;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

SFTTrainer configured without recipe (hyperparameters property).
  learning_rate: 2e-05


## Step 6: Launch Training (Requires AWS Credentials)

When you're ready to run the job, call `.train()`. This submits the training job to SageMaker.

In [9]:
# Launch training
training_job = sft_trainer.train(wait=True)

print(f"Training job completed: {training_job}")

╭────────────────────────────────── Training Job Status ───────────────────────────────────╮
│  TrainingJob Name      nova-textgeneration-lite-v2-sft-20260720181045                    │
│  TrainingJob ARN       arn:aws:sagemaker:us-west-2:674622101542:training-job/nova-textg  │
│                        eneration-lite-v2-sft-20260720181045                              │
│  MLflow Experiment     my-fine-tuned-models                                              │
│  Links                 ]8;id=12572757;https://us-west-2.console.aws.amazon.com/sagemaker/home?region=us-west-2#/jobs/nova-textgeneration-lite-v2-sft-20260720181045\🔗 Training Job (Console)]8;;\                                         │
│                        ]8;id=12572762;https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs$3FlogStreamNameFilter$3Dnova-textgeneration-lite-v2-sft-20260720181045\🔗 CloudWatch Logs]8;;\ | ]8;id=12572763;https://app-3THB6MFIDE4F.mlflow.sagemaker.us-west-2.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IlNQR0NNTCIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNFFLNE5EZEJiVDhUNlpUVHBNRkRNcTFVR2hmUm1BTXozdEpmVEFGczFDTUVBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVFeFVXVmxaRFZIYkRSNGVGaHZhVlJxVFVkSlpFcHBRM0pIWjFwdlZWSkZZVGx3YlZGWGEweFpUekIwVmxwWUwxVnRWamhJZFhwSGVFeFNSbU4wVUZOcVp6MDlBQUVBQjJGM2N5MXJiWE1BUzJGeWJqcGhkM002YTIxek9uVnpMWGRsYzNRdE1qbzVOell4T1RNeU5UUTFOemM2YTJWNUwyVmhNbUZrTUdSa0xXRmtNV0l0TkROalppMDRaak0xTFRNMU5UWmlZVE5pWm1NeE1nQzRBUUlCQUhqRHJwaCtQMVNKcHUzZUl2YnVXUi94bGQ2THMxNjhEU3JZbXo5akp0dlBUQUVmSmhIK1hTSFd4NCtsM3BpTEY0L2hBQUFBZmpCOEJna3Foa2lHOXcwQkJ3YWdiekJ0QWdFQU1HZ0dDU3FHU0liM0RRRUhBVEFlQmdsZ2hrZ0JaUU1FQVM0d0VRUU15M2VuL0dDcGRjbU9rMmZLQWdFUWdEdGFzeDh6L05wRzg3U1ZhM210YlZCY0ltdUs5VDdQQkx5QmFUcVQvb2hXbkE5U0g2RmdhRXRpS3pLL3JOWFhaSUJZek5lR3NHM3FaandDT1FJQUFCQUFvbktqcExmcDg1UGI0TUdHdVRGYmlVbnU3RVE3VEJxbGE5SC9SdEVDV2pnTmVkTDRDR0x6WHl3LzR2K2hIMXFoLy8vLy93QUFBQUVBQUFBQUFBQUFBQUFBQUFFQUFBUWJHaS8xQ3pMQ1ludmR0UVFzb2FUQ2hRMm8zNy9VYzc5T0o5a1MxUm55TzI3aU10QTR1VHhpSEM4RkE3eXVIMStxUWd1cm9oTDFEWUNJSmhOMGVXUlpleUp6MmI3U3RoTkV5cGxtSXgrYVlvUGEzZWdoS0VMbkx6SGt1RjNnemZJazU5M3IyWVFBdkFRZEorWGVJOTdxYUJrUVZHU1FxUi9EU0ViZmpWWndLVHNuQktMZmtlSWJMZ24vMFVJZ04rUzhUK1c2Slh1dFM3ZmNob3FNUzVXb3p2VHFBZFl2Y2picDJEWjliZkNMMGkvN0RidnZxdC94RG1hQURKdDB4Znpqc0FDa1dXQVoxNmJrZnVVRUI3TUY5M0NtL3JjTTBSQTZyVzR5VWpMc1BXRjdjK2dJajRtQnBoblJ2TmUyd25qbEtCdjZiQmJNVjdtbTZBa1JBOGM5NVJHK1RyeWlUWFNKbGtBVEhPN1Z4MHd1VGtpNFFhdjVWTUovdHpqWlpSZDV2cEI1R0FEY1l6YXlpejZzTTZaWHhxbVk0dEI3KzU3VExqSFJJdG50clI5bEgyRHE4aHQ0bFZtWmJNaEtFRjBnNlVQaHNpc3FxV2IzZ2dyZjRIWFdxOVJCcGU3WWdDSWR1ZTFpUzVJYVRzL0xMQnQ1dTVaYXF1NnBhdDBtQkJ2K3hPVUx6NVVYOVdlVDFJTjI4ZXd6cVBhMGIvV3lLdDJnWWZBWU9UbVE3blFuKzgxWTBnclVKQzNFcnJPNVNqekpvcnAvb2g4NmFPSjBXcktDTW9tRjUvcmUvNGFGa2h6VE5Zcytmd2tPWGpEWnRHOVVCSk16QWFHd20vcmNsSFdUcGpiL2xKNVJ0Tmd5TUY4N0tzUTllL3dtdVUvM05xdDhsMVNaOTcwMk5UR1pUVHBwZ0hOK0cvdHdSWGhmYlRiMUJMaTZFL2M2MU1vaVM5Z3pjZi9LTTRqSmlpaGs5VEQ2ZysyZWFrR29jZ3NLSmlGdFNGODVOSkVEZkpKR0lVUzBCRGt6NXpzQjNYc3RPR2JiUndsK0owdVpuR3dkOEdWNU5IS2czN29YT0dUUC9oMGJKalhOcElVbUhVL0ZOT25IK3JvdDVpMW4yTnRKdm1iaTFMSk82U2tPV3ZXbEQxcmNVaFI4NDhDakFVUmZXbStiZGxTQ08zRU9XOEYzZlBWeC8vbDdYa1pTNEpyc2gzKzEzcS82aHVlNkgzRWdEMm81R09zR1FEZjZrUDQxM21sOFQ4cXlya3pTT0JMczU1OFJNdURtbmtqNi9xOEM4bmFMaEVrdHhERmU0SWd3QVUyL3BQd2pvbG5sS1p6YkpKTDJEQ1JQbWVJR2ZjMFhUUDhxTnVscDVwRVZDQjExM3dDbk9Ea3lZcjNxRjhFTjhFcHQycTd0aC9uaXE2N0pCR1FPTEZqb0pFemZjMmFtcUJzeE9SaTVqZ0xvQkc4bmZvcWtoTEZLeXZ6b2llRGlBN0NqMGZqdERnS0Y5ZXdYN1dTZ3dpL1V1czAvQTRkR2FsdThWaWRscjU3cEhkVlI3Qm84OWUwUkJ1TTB0cFN5algyOU1TK2VMN28wNUxkZ2hwK1dKNGVabUo1S3RzWFplNHFmd1o2cFM3OTQwUUJ3OFhVYnBwU3h3Mm5jVU1XcXV3SW1XYXRCZExvM2ExNHd4eVh6dUpQNjF2RzBoWDFRVFM2WEIxRjZpem9TR3FiOGR6MHliWC9hRWpuUGxYcVJDeThNcXNkcW1BTkZ5d29iRWduSStCZzIxb2dZcjVHeXlObVJRZFRLeHVGR1lNUVRpN3RKaXJnNkhxZ1gzUmp1Kzl0NHhuMlVVVnVDekRFZ2FXaVZNNDFyWmJIQUNSSXR6U0ZQdVcwcGNhc1NJQ0I2b2dvQVp6QmxBakE2cXd1bENkTWNDdzVDWnowWUZveXBZcVVjWUhMTmpRaWNYd0NvcnF1U091d0lQTGlUK1JjZ3pHSEN

Training job completed: training_job_name='nova-textgeneration-lite-v2-sft-20260720181045' training_job_arn='arn:aws:sagemaker:us-west-2:674622101542:training-job/nova-textgeneration-lite-v2-sft-20260720181045' tuning_job_arn=Unassigned() labeling_job_arn=Unassigned() auto_ml_job_arn=Unassigned() model_artifacts=Unassigned() training_job_status='Completed' secondary_status='Completed' failure_reason=Unassigned() hyper_parameters={'fine_tuned_model': '1.0', 'global_batch_size': '32', 'learning_rate': '1e-05', 'learning_rate_ratio': '64.0', 'limit_val_batches': '2', 'lora_alpha': '64', 'max_context_length': '8192', 'max_steps': '20', 'min_lr': '1e-06', 'name': 'my-lora-sft-run-64qsr', 'reasoning_enabled': 'True', 'save_steps': '10', 'val_check_interval': '2500', 'warmup_steps': '15', 'weight_decay': '0.0'} algorithm_specification=Unassigned() role_arn='arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole' input_data_config=[Channel(channel_name='train', data_source=DataSou

## Summary

The recipe override system provides:

- **Reusability** — Define base configs in YAML files, share across team
- **Flexibility** — Override any parameter at runtime without editing the file
- **Full recipe access** — Power users can override **any** key in the full Hub recipe, not just the UI-exposed subset (e.g., `sequence_length`, `warmup_ratio`, `gradient_accumulation_steps`)
- **Transparency** — `get_resolved_recipe()` shows exactly what will be used
- **Backward compatibility** — The `hyperparameters` property still works

**Precedence (highest to lowest):**
1. `overrides` dict
2. Recipe YAML file
3. Full Hub recipe defaults (all parameters)
